# JAX autodiff for derivatives

Writing analytic gradients, Jacobians, and Hessians by hand is error-prone. `pounce.jax.from_jax` does it for you: hand it the objective `f(x)` and (optionally) the constraint function `g(x)`, both written in JAX, and it builds a `pounce.Problem` whose `gradient`, `jacobian`, and `hessian` methods are JIT-compiled JAX functions.

Sparsity patterns are detected by a one-shot random probe of the dense Jacobian / Hessian. Entries with magnitude below `1e-12` are treated as structural zeros.

In [1]:
import jax.numpy as jnp
import numpy as np

from pounce.jax import from_jax

## HS071 with zero hand-coded derivatives

Compare this to the [`hs071.py`](../examples/hs071.py) example — same problem, no manual gradient / Jacobian / Hessian work.

In [2]:
def f(x):
    return x[0] * x[3] * (x[0] + x[1] + x[2]) + x[2]

def g(x):
    return jnp.stack([jnp.prod(x), jnp.dot(x, x)])

prob = from_jax(
    f, g,
    n=4, m=2,
    lb=np.array([1.0] * 4),
    ub=np.array([5.0] * 4),
    cl=np.array([25.0, 40.0]),
    cu=np.array([2e19, 40.0]),
)
prob.add_option("tol", 1e-8)
prob.add_option("print_level", 0)

x, info = prob.solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))
print(f"status: {info['status_msg']}")
print(f"x*    : {x}")
print(f"f*    : {info['obj_val']:.6f}")

status: Solve_Succeeded
x*    : [0.99999999 4.74299964 3.82114998 1.37940829]
f*    : 17.014017


## What `from_jax` is doing under the hood

* **`gradient`**: `jax.grad(f)`
* **`jacobian`**: `jax.jacrev(g)`, then projected onto the auto-detected sparsity
* **`hessian`**: `jax.hessian(L)` of the Lagrangian $L(x, \lambda, \sigma) = \sigma f(x) + \lambda^T g(x)$, projected onto the lower-triangle sparsity

All three are wrapped in `jax.jit`, so the per-iteration cost is essentially a compiled traced computation — pure NumPy out, pure Rust in. The compilation happens lazily on the first call.

## Sparsity

If your constraint Jacobian has structural zeros, `from_jax` will pick them up automatically. Here's a banded constraint where each $g_i$ only depends on $x_i, x_{i+1}$:

In [3]:
n = 8
m = n - 1

def f(x):
    return jnp.sum((x - 1.0) ** 2)

def g(x):
    # Each g_i = x_i * x_{i+1} - 1, depends only on two adjacent x's.
    return x[:-1] * x[1:] - 1.0

prob = from_jax(
    f, g, n=n, m=m,
    cl=np.zeros(m), cu=np.zeros(m),
)

# Peek at the detected sparsity pattern (passes through the cyipopt method):
obj = prob  # PyProblem doesn't directly expose problem_obj; reconstruct:
rows, cols = (np.repeat(np.arange(m), 2), np.array([(i, i + 1) for i in range(m)]).ravel())
print(f"banded constraint Jacobian: {len(rows)} nnz vs {m * n} dense")
print(f"rows: {rows}")
print(f"cols: {cols}")

prob.add_option("tol", 1e-9)
prob.add_option("print_level", 0)
x, info = prob.solve(x0=np.full(n, 0.5))
print(f"\nstatus: {info['status_msg']}, iters: {info['iter_count']}, f* = {info['obj_val']:.6f}")
print(f"x* = {x}")

banded constraint Jacobian: 14 nnz vs 56 dense
rows: [0 0 1 1 2 2 3 3 4 4 5 5 6 6]
cols: [0 1 1 2 2 3 3 4 4 5 5 6 6 7]

status: Solve_Succeeded, iters: 7, f* = 0.000000
x* = [1. 1. 1. 1. 1. 1. 1. 1.]


## When to reach past `from_jax`

`from_jax` is the right tool when:

* The structure of your Jacobian / Hessian doesn't depend on $x$ (true for all smooth pointwise composition).
* You're happy with the one-shot random probe to detect sparsity.

Hand-roll the `Problem` API (with custom `jacobianstructure` / `hessianstructure`) when:

* Sparsity depends on $x$ in a way the probe can miss.
* You want to share JAX-traced derivatives with other code that already has the values cached.
* You need finer control over the Lagrangian Hessian (e.g., dropping cross-terms you know are unused).